# 제13장. 실물옵션 분석## 3시간 인터랙티브 강의 (이론 + 실습)**학습 목표:**1. 금융옵션과 실물옵션의 차이를 이해하고 **유연성의 경제적 가치**를 설명할 수 있다2. 실물옵션의 유형(연기/확장/포기/전환/단계적 투자)을 **식별**할 수 있다3. **이항 모델**로 실물옵션 가치를 계산할 수 있다4. **NPV의 한계**를 이해하고 확장 NPV 개념을 적용할 수 있다5. **단계적 투자**의 옵션 가치를 시뮬레이션으로 평가할 수 있다**강의 구조:**| 시간 | Part | 내용 ||------|------|------|| | **Part 1** | 📖 실물옵션 기초: 금융옵션 → 실물옵션, NPV 한계, 유연성 가치 + 조사 과제 || | **Part 2** | 📖 실물옵션 유형과 가치평가: 연기/확장/포기/전환, 이항 모델 + 조사 과제 || | **휴식** | ☕ 15분 휴식 || | **Part 3** | 📖 단계적 투자와 기획 적용: Stage-Gate, R&D, 전략적 유연성 + 조사 과제 || | **Part 4** | 🔬 실습: 이항 모델 옵션 가치 계산 || | **Part 5** | 🔬 실습: 단계적 투자 시뮬레이션 |---

In [ ]:
# ============================================================
# 환경설정 및 라이브러리 로드
# ============================================================
# 처음 사용 시 아래 명령을 터미널에서 먼저 실행하세요:
#   python setup_env.py
# 그 후 VSCode 우측 상단에서 커널을 'AI 기획 강의 (Python 3)'으로 선택하세요.
# ============================================================

import sys

# 패키지 확인
_required = ["numpy", "pandas", "plotly", "scipy"]
_missing = []
for _pkg in _required:
    try:
        __import__(_pkg)
    except ImportError:
        _missing.append(_pkg)
if _missing:
    print(f"❌ 누락된 패키지: {', '.join(_missing)}")
    print("   터미널에서 다음 명령을 실행하세요: python setup_env.py")
    raise SystemExit(1)

# 라이브러리 임포트
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy import stats

import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print("✅ 라이브러리 로드 완료!")

---## Part 1: 📖 실물옵션 기초### 1.1 금융옵션의 기본 개념**옵션(Option)**은 특정 자산을 미래의 정해진 가격에 매수/매도할 수 있는 **권리**이다.핵심은 "의무"가 아닌 **"권리"**라는 점이다.| 구분 | 콜옵션 (Call) | 풋옵션 (Put) ||------|--------------|-------------|| **권리** | 기초자산 매수 권리 | 기초자산 매도 권리 || **행사 조건** | 시장가 > 행사가 | 시장가 < 행사가 || **최대 손실** | 옵션 프리미엄 | 옵션 프리미엄 || **최대 이익** | 무한 | 행사가 - 0 |### 1.2 옵션 가치의 5가지 결정 요인1. **기초자산 현재가치** (S₀): 높을수록 콜옵션 가치 ↑2. **행사가격** (K): 낮을수록 콜옵션 가치 ↑3. **만기까지 시간** (T): 길수록 옵션 가치 ↑4. **변동성** (σ): 높을수록 옵션 가치 ↑ ← **핵심!**5. **무위험 이자율** (r): 높을수록 콜옵션 가치 ↑> **핵심 통찰:** 변동성이 높을수록 옵션 가치가 높아진다. 불확실성이 옵션에게는 **기회**이기 때문이다.### 1.3 금융옵션에서 실물옵션으로| 요소 | 금융옵션 (콜옵션) | 실물옵션 (투자 옵션) ||------|------------------|---------------------|| 기초자산 | 주식, 채권 등 | 프로젝트, 사업 || 현재 가치 | 주가 | 프로젝트 현재가치 || 행사가격 | 옵션 행사가 | 투자비용 || 만기 | 옵션 만기일 | 투자 기회 만료일 || 변동성 | 주가 변동성 | 프로젝트 가치 변동성 || 옵션 행사 | 주식 매수 | 투자 실행 |### 1.4 NPV의 한계와 확장 NPV전통적 NPV는 **"지금 결정하고, 그대로 실행한다"**는 정적 관점을 취한다.$$\text{확장 NPV} = \text{전통적 NPV} + \text{실물옵션 가치}$$- 전통적 NPV가 **음수**여도 옵션 가치가 크면 투자 정당화 가능- 전통적 NPV가 **양수**여도 "기다릴 권리"의 가치가 크면 연기가 최적---

In [ ]:
# ========================================# 콜옵션 페이오프 다이어그램# ========================================S = np.linspace(50, 200, 200)K = 100  # 행사가격premium = 15  # 옵션 프리미엄# 콜옵션 페이오프payoff_long = np.maximum(S - K, 0) - premiumpayoff_short = -(np.maximum(S - K, 0) - premium)fig = make_subplots(rows=1, cols=2,                    subplot_titles=['Call Option (Long)', 'Put Option Payoff'])# 콜옵션 매수자fig.add_trace(go.Scatter(x=S, y=payoff_long, mode='lines',                         name='Long Call', line=dict(color='#2ecc71', width=3)), row=1, col=1)fig.add_hline(y=0, line_dash='dash', line_color='gray', row=1, col=1)fig.add_vline(x=K, line_dash='dot', line_color='red', row=1, col=1,              annotation_text=f'Strike K={K}')# 풋옵션 페이오프payoff_put = np.maximum(K - S, 0) - premiumfig.add_trace(go.Scatter(x=S, y=payoff_put, mode='lines',                         name='Long Put', line=dict(color='#e74c3c', width=3)), row=1, col=2)fig.add_hline(y=0, line_dash='dash', line_color='gray', row=1, col=2)fig.add_vline(x=K, line_dash='dot', line_color='red', row=1, col=2,              annotation_text=f'Strike K={K}')fig.update_layout(title='Option Payoff Diagrams', height=400, width=900,                  showlegend=True)fig.update_xaxes(title_text='Asset Price (S)', row=1, col=1)fig.update_xaxes(title_text='Asset Price (S)', row=1, col=2)fig.update_yaxes(title_text='Profit/Loss', row=1, col=1)fig.update_yaxes(title_text='Profit/Loss', row=1, col=2)fig.show()print("💡 콜옵션: 기초자산 가격이 행사가(100)를 넘으면 이익, 최대 손실은 프리미엄(15)")print("💡 풋옵션: 기초자산 가격이 행사가(100) 미만이면 이익")print("💡 실물옵션에서 '투자할 권리'는 콜옵션, '포기할 권리'는 풋옵션과 유사")

In [ ]:
# ========================================# NPV vs 확장 NPV 비교# ========================================projects = ['Project A\n(R&D)', 'Project B\n(Market Entry)', 'Project C\n(Infrastructure)']npv_traditional = [-10, 10, 30]option_values = [25, 14, 8]expanded_npv = [n + o for n, o in zip(npv_traditional, option_values)]fig = go.Figure()fig.add_trace(go.Bar(name='Traditional NPV', x=projects, y=npv_traditional,                     marker_color='#3498db', text=[f'{v:+.0f}B' for v in npv_traditional],                     textposition='auto'))fig.add_trace(go.Bar(name='Option Value', x=projects, y=option_values,                     marker_color='#f39c12', text=[f'+{v:.0f}B' for v in option_values],                     textposition='auto'))fig.add_trace(go.Scatter(name='Expanded NPV', x=projects, y=expanded_npv,                         mode='markers+text', marker=dict(size=16, color='#e74c3c', symbol='diamond'),                         text=[f'{v:+.0f}B' for v in expanded_npv], textposition='top center'))fig.add_hline(y=0, line_dash='dash', line_color='gray', line_width=2)fig.update_layout(title='Traditional NPV vs Expanded NPV (100M KRW)',                  barmode='stack', height=450, width=700,                  yaxis_title='Value (100M KRW)')fig.show()print("📊 분석 결과:")print(f"  Project A: 전통 NPV {npv_traditional[0]:+}억 (기각) → 확장 NPV {expanded_npv[0]:+}억 (투자 가치!)")print(f"  Project B: 전통 NPV {npv_traditional[1]:+}억 → 확장 NPV {expanded_npv[1]:+}억 (옵션으로 가치 상승)")print(f"  Project C: 전통 NPV {npv_traditional[2]:+}억 → 확장 NPV {expanded_npv[2]:+}억 (옵션 가치 상대적 작음)")print("\n💡 핵심: NPV가 음수인 Project A도 유연성(옵션)을 고려하면 투자 가치가 있다!")

### 📝 이론 과제 13-1 (15분)#### 과제: 실물옵션의 기본 개념**질문:**1. **금융옵션**과 **실물옵션**의 공통점과 차이점을 설명하라. 특히 "기초자산"과 "행사"의 의미가 어떻게 다른지 비교하라. (3-4문장)2. NPV가 양수(+20억)인 프로젝트인데도 **투자를 연기하는 것이 최적**인 상황은 어떤 경우인가? 예시와 함께 설명하라. (3-4문장)3. 본인이 관심 있는 산업/기업에서 **실물옵션**이 존재하는 사례를 1가지 찾아 설명하라. 어떤 유형의 옵션(연기/확장/포기 등)에 해당하는가?**조사 키워드:**- "real options vs financial options"- "NPV limitations flexibility value"- "real options in strategic planning"**제출:** 아래 마크다운 셀에 **직접 타이핑** (복사 붙여넣기 금지)---

### ✅ 과제 13-1 제출란**학번:** ___________**이름:** ___________#### 1. 금융옵션 vs 실물옵션(여기에 작성)#### 2. NPV 양수인데 연기가 최적인 상황(여기에 작성)#### 3. 실물옵션 사례- 산업/기업:- 옵션 유형:- 설명:**출처:** (URL 또는 문헌)---

---## Part 2: 📖 실물옵션 유형과 가치평가### 2.1 실물옵션의 5가지 유형| 옵션 유형 | 유사 금융옵션 | 적용 사례 | 핵심 가치 ||----------|-------------|----------|----------|| **연기 옵션** | 미국형 콜 | R&D, 시장 진입 | 정보 수집 후 결정 || **확장 옵션** | 콜옵션 | 파일럿 → 본격 투자 | 성공 시 규모 확대 || **축소/포기 옵션** | 풋옵션 | 광산, M&A | 실패 시 손실 제한 || **전환 옵션** | 스왑션 | 유연 생산, 복합 설비 | 시장 변화에 적응 || **단계적 투자** | 복합 옵션 | 신약 개발, 벤처 | 관문별 중단 가능 |### 2.2 이항 모델(Binomial Model) 기초가장 직관적인 옵션 가치 평가 방법. 자산 가치가 각 시점에서 **상승** 또는 **하락**만 가능하다고 가정.**핵심 파라미터:**- 상승 비율: $u = e^{\sigma\sqrt{\Delta t}}$- 하락 비율: $d = 1/u$- 위험중립 확률: $p = \frac{e^{r\Delta t} - d}{u - d}$**역방향 귀납법(Backward Induction):**1. 만기 시점에서 옵션 내재가치 계산: max(S - K, 0)2. 각 노드에서: max(즉시 행사 가치, 계속 보유 가치)3. 계속 보유 가치 = (p × 상승가치 + (1-p) × 하락가치) × e^(-rΔt)### 2.3 블랙-숄즈 모델 (참고)$$C = S_0 N(d_1) - K e^{-rT} N(d_2)$$이항 모델에서 시간 간격을 무한히 작게 하면 블랙-숄즈에 수렴. 실물옵션에는 이항 모델이 더 직관적.---

In [ ]:
# ========================================# 실물옵션 유형별 가치 특성 비교# ========================================option_types = ['Defer', 'Expand', 'Contract/\nAbandon', 'Switch', 'Staged\nInvestment']uncertainty_value = [5, 4, 3, 4, 5]  # 불확실성이 높을수록 가치flexibility_level = [4, 4, 3, 5, 5]implementation_ease = [4, 3, 3, 2, 4]fig = go.Figure()fig.add_trace(go.Scatterpolar(    r=uncertainty_value + [uncertainty_value[0]],    theta=option_types + [option_types[0]],    fill='toself', name='Uncertainty Value',    line=dict(color='#e74c3c'), opacity=0.6))fig.add_trace(go.Scatterpolar(    r=flexibility_level + [flexibility_level[0]],    theta=option_types + [option_types[0]],    fill='toself', name='Flexibility Level',    line=dict(color='#3498db'), opacity=0.6))fig.add_trace(go.Scatterpolar(    r=implementation_ease + [implementation_ease[0]],    theta=option_types + [option_types[0]],    fill='toself', name='Implementation Ease',    line=dict(color='#2ecc71'), opacity=0.6))fig.update_layout(    polar=dict(radialaxis=dict(visible=True, range=[0, 5])),    title='Real Option Types - Characteristic Comparison',    height=500, width=600)fig.show()print("📊 실물옵션 유형별 특성:")print("  - 연기 옵션: 불확실성 가치 최고, 정보 수집 후 결정")print("  - 단계적 투자: 유연성과 불확실성 가치 모두 높음")print("  - 전환 옵션: 유연성 최고, 구현 난이도도 높음")

In [ ]:
# ========================================# 이항 모델 계산 및 시각화 (2기간)# ========================================def binomial_option_value(S0, K, sigma, r, T, n=2, option_type='call'):    """이항 모델로 실물옵션 가치 계산"""    dt = T / n    u = np.exp(sigma * np.sqrt(dt))    d = 1 / u    p = (np.exp(r * dt) - d) / (u - d)    discount = np.exp(-r * dt)    # 자산 가격 트리    asset_tree = np.zeros((n + 1, n + 1))    for i in range(n + 1):        for j in range(i + 1):            asset_tree[j, i] = S0 * (u ** (i - j)) * (d ** j)    # 옵션 가치 트리    option_tree = np.zeros((n + 1, n + 1))    for j in range(n + 1):        if option_type == 'call':            option_tree[j, n] = max(asset_tree[j, n] - K, 0)        else:            option_tree[j, n] = max(K - asset_tree[j, n], 0)    for i in range(n - 1, -1, -1):        for j in range(i + 1):            hold = discount * (p * option_tree[j, i + 1] + (1 - p) * option_tree[j + 1, i + 1])            if option_type == 'call':                exercise = max(asset_tree[j, i] - K, 0)            else:                exercise = max(K - asset_tree[j, i], 0)            option_tree[j, i] = max(hold, exercise)    return asset_tree, option_tree, u, d, p# 파라미터 설정S0, K, sigma, r, T = 100, 90, 0.30, 0.03, 2asset_tree, option_tree, u, d, p = binomial_option_value(S0, K, sigma, r, T, n=2)print(f"📊 이항 모델 파라미터")print(f"  S₀={S0}억, K={K}억, σ={sigma:.0%}, r={r:.0%}, T={T}년")print(f"  u = {u:.4f}, d = {d:.4f}, p = {p:.4f}")print(f"\n자산 가격 트리 (억원):")for i in range(3):    vals = [f"{asset_tree[j,i]:.1f}" for j in range(i+1)]    print(f"  t={i}: {', '.join(vals)}")print(f"\n옵션 가치 트리 (억원):")for i in range(3):    vals = [f"{option_tree[j,i]:.1f}" for j in range(i+1)]    print(f"  t={i}: {', '.join(vals)}")print(f"\n🎯 결과:")print(f"  전통적 NPV = S₀ - K = {S0} - {K} = {S0-K}억 원")print(f"  실물옵션 가치 = {option_tree[0,0]:.2f}억 원")print(f"  유연성 가치 = {option_tree[0,0] - (S0-K):.2f}억 원")# 시각화fig = go.Figure()positions = {(0,0): (0, 0), (0,1): (1, 1), (1,1): (1, -1),             (0,2): (2, 2), (1,2): (2, 0), (2,2): (2, -2)}for (j, i), (x, y) in positions.items():    fig.add_trace(go.Scatter(        x=[x], y=[y], mode='markers+text',        marker=dict(size=50, color='#3498db' if i < 2 else ('#2ecc71' if option_tree[j,i] > 0 else '#e74c3c')),        text=[f"S={asset_tree[j,i]:.1f}<br>V={option_tree[j,i]:.1f}"],        textposition='middle center', textfont=dict(size=10, color='white'),        showlegend=False))# 연결선edges = [((0,0),(0,1)), ((0,0),(1,1)), ((0,1),(0,2)), ((0,1),(1,2)), ((1,1),(1,2)), ((1,1),(2,2))]for (j1,i1), (j2,i2) in edges:    x1, y1 = positions[(j1,i1)]    x2, y2 = positions[(j2,i2)]    fig.add_trace(go.Scatter(x=[x1,x2], y=[y1,y2], mode='lines',                             line=dict(color='gray', width=1), showlegend=False))fig.update_layout(title=f'Binomial Tree (S₀={S0}, K={K}, σ={sigma:.0%}, T={T}yr)',                  xaxis_title='Time Period', yaxis_title='',                  height=400, width=600, showlegend=False,                  xaxis=dict(tickvals=[0,1,2], ticktext=['t=0','t=1','t=2']))fig.show()

In [ ]:
# ========================================# 변동성 민감도 분석# ========================================volatilities = np.arange(0.10, 0.55, 0.05)option_vals = []flex_vals = []npv_base = S0 - Kfor vol in volatilities:    _, opt_tree, _, _, _ = binomial_option_value(S0, K, vol, r, T, n=10)    ov = opt_tree[0, 0]    option_vals.append(ov)    flex_vals.append(ov - npv_base)fig = go.Figure()fig.add_trace(go.Scatter(x=volatilities*100, y=option_vals,                         mode='lines+markers', name='Option Value',                         line=dict(color='#e74c3c', width=3)))fig.add_trace(go.Scatter(x=volatilities*100, y=flex_vals,                         mode='lines+markers', name='Flexibility Value',                         line=dict(color='#3498db', width=3)))fig.add_hline(y=npv_base, line_dash='dash', line_color='gray',              annotation_text=f'Traditional NPV = {npv_base}')fig.update_layout(title='Option Value vs Volatility',                  xaxis_title='Volatility (%)', yaxis_title='Value (100M KRW)',                  height=400, width=700)fig.show()print("📊 변동성 민감도 분석:")print(f"  변동성 10%: 옵션가치 {option_vals[0]:.1f}억, 유연성가치 {flex_vals[0]:.1f}억")print(f"  변동성 30%: 옵션가치 {option_vals[4]:.1f}억, 유연성가치 {flex_vals[4]:.1f}억")print(f"  변동성 50%: 옵션가치 {option_vals[8]:.1f}억, 유연성가치 {flex_vals[8]:.1f}억")print("\n💡 핵심: 변동성이 높을수록 옵션 가치(유연성 가치)가 증가한다!")print("   → 불확실성이 높은 프로젝트에서 실물옵션 분석이 더 유용하다")

### 📝 이론 과제 13-2 (15분)#### 과제: 실물옵션 유형과 이항 모델**질문:**1. 5가지 실물옵션 유형(연기/확장/축소·포기/전환/단계적 투자) 중 **3가지**를 선택하여, 각각의 의미와 실제 적용 사례를 설명하라. (각 2-3문장)2. 이항 모델에서 **위험중립 확률(Risk-Neutral Probability)**을 사용하는 이유를 조사하라. 실제 확률과 무엇이 다른가? (3-4문장)3. "변동성이 높을수록 옵션 가치가 증가한다"는 결론이 직관에 반하는 것처럼 보일 수 있다. 왜 불확실성이 옵션 보유자에게 유리한지 설명하라. (3-4문장)**조사 키워드:**- "binomial option pricing model"- "risk-neutral valuation real options"- "volatility option value relationship"**제출:** 아래 마크다운 셀에 **직접 타이핑** (복사 붙여넣기 금지)---

### ✅ 과제 13-2 제출란**학번:** ___________**이름:** ___________#### 1. 실물옵션 유형 3가지(여기에 작성)#### 2. 위험중립 확률을 사용하는 이유(여기에 작성)#### 3. 변동성과 옵션 가치의 관계(여기에 작성)**출처:** (URL 또는 문헌)---

---## ☕ 휴식 (15분)15분 휴식 후 **Part 3**에서 계속됩니다.---

---## Part 3: 📖 단계적 투자와 기획 적용### 3.1 단계적 투자(Staged Investment)프로젝트를 여러 단계로 나누어 순차 투자. 각 단계 완료 후 **계속/중단**을 결정.이는 **복합 옵션(Compound Option)**으로 모델링.**신약 개발 R&D 사례:**| 단계 | 투자금액 | 기간 | 성공률 | 내용 ||------|---------|------|-------|------|| 1단계 | 20억 원 | 2년 | 70% | 전임상/기초연구 || 2단계 | 50억 원 | 2년 | 60% | 임상1상/프로토타입 || 3단계 | 100억 원 | 3년 | 50% | 임상2상/파일럿 || 4단계 | 200억 원 | 3년 | 70% | 임상3상/상용화 || **합계** | **370억 원** | **10년** | **14.7%** | |최종 성공 시 프로젝트 가치: **1,000억 원**### 3.2 Stage-Gate 프로세스| 관문 | 평가 기준 | 옵션 결정 | 예상 중단율 ||------|----------|----------|-----------|| Gate 0 | 아이디어 스크리닝 | Go/Kill | 60% || Gate 1 | 타당성 검토 | Go/Kill/Hold | 30% || Gate 2 | 개발 착수 | Go/Kill/Recycle | 20% || Gate 3 | 시험 생산 | Go/Kill | 15% || Gate 4 | 상업화 | Launch/Kill | 10% |### 3.3 전략적 유연성 설계옵션을 **의도적으로 설계**하는 4가지 원칙:1. **모듈화**: 독립적 모듈로 분리 → 개별 확장/축소2. **단계화**: 대규모 투자를 단계로 분할 → 각 단계에서 재평가3. **옵션 계약**: 공급 계약, 부동산 옵션으로 미래 권리 확보4. **플랫폼 투자**: 다양한 후속 사업의 기반이 되는 인프라### 3.4 실물옵션 분석의 한계- **변동성 추정 어려움**: 프로젝트 가치의 변동성은 추정이 어려움- **과대평가 위험**: 옵션 행사 비용, 경쟁자 반응 무시 가능- **정성적 판단 필요**: 전략적 적합성, 조직 역량은 모델에 미반영---

In [ ]:
# ========================================# 단계적 투자 vs 일괄 투자 몬테카를로 시뮬레이션# ========================================def simulate_staged_investment(n_sims=10000, discount_rate=0.08):    """단계적 투자 시뮬레이션"""    stages = [        {'name': 'Stage 1 (Preclinical)', 'cost': 20, 'duration': 2, 'success_rate': 0.70},        {'name': 'Stage 2 (Phase 1)', 'cost': 50, 'duration': 2, 'success_rate': 0.60},        {'name': 'Stage 3 (Phase 2)', 'cost': 100, 'duration': 3, 'success_rate': 0.50},        {'name': 'Stage 4 (Phase 3)', 'cost': 200, 'duration': 3, 'success_rate': 0.70},    ]    final_value = 1000    total_cost = sum(s['cost'] for s in stages)    total_duration = sum(s['duration'] for s in stages)    # 전통적 NPV (일괄 투자)    cum_prob = 1.0    for s in stages:        cum_prob *= s['success_rate']    expected_value = cum_prob * final_value / (1 + discount_rate) ** total_duration    traditional_npv = expected_value - total_cost    # 단계적 투자 시뮬레이션    np.random.seed(42)    payoffs = []    stage_reach = [0] * (len(stages) + 1)    for _ in range(n_sims):        total_invested = 0        years_elapsed = 0        reached_end = True        for i, stage in enumerate(stages):            total_invested += stage['cost']            years_elapsed += stage['duration']            if np.random.random() > stage['success_rate']:                payoffs.append(-total_invested / (1 + discount_rate) ** years_elapsed)                reached_end = False                break            stage_reach[i + 1] += 1        if reached_end:            pv = final_value / (1 + discount_rate) ** years_elapsed            payoffs.append(pv - total_invested / (1 + discount_rate) ** years_elapsed)    stage_reach[0] = n_sims    option_value = np.mean(payoffs)    flexibility_value = option_value - traditional_npv    return {        'traditional_npv': traditional_npv,        'option_value': option_value,        'flexibility_value': flexibility_value,        'payoffs': payoffs,        'stage_reach': stage_reach,        'stages': stages,        'n_sims': n_sims,        'cum_prob': cum_prob    }results = simulate_staged_investment()print("📊 단계적 투자 시뮬레이션 결과 (10,000회)")print("=" * 60)print(f"  전통적 NPV (일괄투자): {results['traditional_npv']:.2f}억 원")print(f"  단계적 투자 옵션 가치: {results['option_value']:.2f}억 원")print(f"  유연성 가치:           {results['flexibility_value']:.2f}억 원")print(f"  누적 성공률:           {results['cum_prob']:.1%}")print(f"\n단계별 도달률:")for i, s in enumerate(results['stages']):    rate = results['stage_reach'][i+1] / results['n_sims'] * 100    print(f"  {s['name']}: {rate:.1f}% ({results['stage_reach'][i+1]:,}건)")

In [ ]:
# ========================================# 시각화: 단계별 생존 퍼널 + 결과 분포# ========================================fig = make_subplots(rows=1, cols=2,    subplot_titles=['Stage Survival Funnel', 'Payoff Distribution (Staged)'])# 퍼널stage_names = ['Start'] + [s['name'].split(' (')[0] for s in results['stages']]reach_rates = [r / results['n_sims'] * 100 for r in results['stage_reach']]fig.add_trace(go.Funnel(    y=stage_names, x=reach_rates,    textinfo='value+percent initial',    marker=dict(color=['#3498db', '#2ecc71', '#f1c40f', '#e67e22', '#e74c3c'])), row=1, col=1)# 페이오프 분포fig.add_trace(go.Histogram(    x=results['payoffs'], nbinsx=50,    marker_color='#3498db', opacity=0.7,    name='Payoff'), row=1, col=2)fig.add_vline(x=0, line_dash='dash', line_color='red', row=1, col=2)fig.add_vline(x=results['option_value'], line_dash='solid', line_color='green',              row=1, col=2, annotation_text=f"Mean={results['option_value']:.1f}")fig.update_layout(height=400, width=1000,                  title='Staged Investment Analysis')fig.update_xaxes(title_text='Survival Rate (%)', row=1, col=1)fig.update_xaxes(title_text='Payoff (100M KRW)', row=1, col=2)fig.show()print("💡 퍼널 해석: 각 관문에서 적극적으로 중단 옵션이 행사됨")print("   → 1단계 실패 시 20억만 손실, 370억 전액 투자 대비 리스크 대폭 감소")

In [ ]:
# ========================================# 포트폴리오 관점: 동시 추진 프로젝트 수# ========================================single_success = results['stage_reach'][-1] / results['n_sims']n_projects = [1, 3, 5, 10, 15, 20]portfolio_success = [1 - (1 - single_success) ** n for n in n_projects]stage1_cost = results['stages'][0]['cost']total_costs = [n * stage1_cost for n in n_projects]expected_successes = [n * single_success for n in n_projects]fig = make_subplots(rows=1, cols=2,    subplot_titles=['Portfolio Success Probability', 'Expected Successes vs Stage 1 Cost'])fig.add_trace(go.Bar(    x=[str(n) for n in n_projects], y=[p*100 for p in portfolio_success],    marker_color='#2ecc71',    text=[f'{p:.1%}' for p in portfolio_success], textposition='auto'), row=1, col=1)fig.add_trace(go.Bar(    x=[str(n) for n in n_projects], y=expected_successes,    marker_color='#3498db',    text=[f'{e:.1f}' for e in expected_successes], textposition='auto'), row=1, col=2)fig.update_layout(height=400, width=900, title='Portfolio Perspective on R&D Investment',                  showlegend=False)fig.update_xaxes(title_text='Number of Projects', row=1, col=1)fig.update_xaxes(title_text='Number of Projects', row=1, col=2)fig.update_yaxes(title_text='At Least 1 Success (%)', row=1, col=1)fig.update_yaxes(title_text='Expected Successes', row=1, col=2)fig.show()print(f"📊 포트폴리오 분석 (단일 프로젝트 성공률: {single_success:.1%})")for n, ps, tc in zip(n_projects, portfolio_success, total_costs):    print(f"  {n:>2}개 동시: 1단계 비용 {tc:>4}억, 최소 1개 성공 확률 {ps:.1%}")print("\n💡 이것이 벤처캐피탈이 다수 스타트업에 분산 투자하는 이유!")

### 📝 이론 과제 13-3 (15분)#### 과제: 단계적 투자와 기획 적용**질문:**1. **단계적 투자**가 일괄 투자보다 가치 있는 조건 3가지를 설명하라. (3-4문장)2. **Stage-Gate 프로세스**의 장점과, 관문에서 "어쨌든 계속하자"는 태도가 왜 위험한지 설명하라. (3-4문장)3. 본인이 관심 있는 분야에서 **단계적 투자**를 적용할 수 있는 프로젝트를 제안하라. 각 단계의 투자금액, 기간, 성공 기준을 간략히 설계하라.**조사 키워드:**- "staged investment real options"- "Stage-Gate process Cooper"- "venture capital portfolio strategy"**제출:** 아래 마크다운 셀에 **직접 타이핑**---

### ✅ 과제 13-3 제출란**학번:** ___________**이름:** ___________#### 1. 단계적 투자가 유리한 조건(여기에 작성)#### 2. Stage-Gate의 장점과 위험(여기에 작성)#### 3. 단계적 투자 프로젝트 설계- 프로젝트명:- 1단계: 금액 / 기간 / 성공 기준- 2단계: 금액 / 기간 / 성공 기준- 3단계: 금액 / 기간 / 성공 기준**출처:** (URL 또는 문헌)---

---## Part 4: 🔬 실습 - 이항 모델 옵션 가치 계산### 실습 배경**시나리오:** 신규 AI 플랫폼 시장 진출 투자| 파라미터 | 값 | 의미 ||---------|-----|------|| S₀ | 200억 원 | 프로젝트 현재가치 || K | 180억 원 | 투자비용 || σ | 35% | 가치 변동성 || r | 3% | 무위험 이자율 || T | 3년 | 의사결정 기간 |**질문:** 즉시 투자(NPV=20억)와 연기 옵션 중 어느 것이 최적인가?---

In [ ]:
# ========================================# 연기 옵션 분석: 즉시 투자 vs 기다림# ========================================S0_d, K_d, sigma_d, r_d, T_d = 200, 180, 0.35, 0.03, 3# 다양한 기간에 대한 옵션 가치periods = [1, 2, 3, 4, 5]defer_values = []for t in periods:    _, opt, _, _, _ = binomial_option_value(S0_d, K_d, sigma_d, r_d, t, n=max(t*2, 4))    defer_values.append(opt[0, 0])npv_immediate = S0_d - K_d  # 20억print("📊 연기 옵션 분석")print(f"  즉시 투자 NPV = {npv_immediate}억 원")print(f"\n연기 기간별 옵션 가치:")print(f"  {'기간':>6} {'옵션가치':>10} {'연기의가치':>12} {'권고':>10}")print("  " + "-" * 45)for t, dv in zip(periods, defer_values):    defer_gain = dv - npv_immediate    rec = '연기 권장' if defer_gain > 0 else '즉시 투자'    print(f"  {t}년   {dv:>10.2f}억   {defer_gain:>10.2f}억   {rec}")# 시각화fig = go.Figure()fig.add_trace(go.Bar(x=[f'{t}yr' for t in periods], y=defer_values,                     name='Defer Option Value', marker_color='#3498db',                     text=[f'{v:.1f}B' for v in defer_values], textposition='auto'))fig.add_hline(y=npv_immediate, line_dash='dash', line_color='red', line_width=3,              annotation_text=f'Immediate NPV = {npv_immediate}B')fig.update_layout(title=f'Defer Option Value by Period (S₀={S0_d}, K={K_d}, σ={sigma_d:.0%})',                  xaxis_title='Deferral Period', yaxis_title='Value (100M KRW)',                  height=400, width=600)fig.show()print(f"\n💡 핵심: 즉시 투자 NPV가 {npv_immediate}억(양수)임에도, 기다리는 것이 더 가치 있다!")print("   → 불확실성이 해소될 때까지 기다렸다가 유리할 때만 투자하는 전략")

In [ ]:
# ========================================# 민감도 분석: 변동성 × 프로젝트가치비율# ========================================vols = [0.10, 0.20, 0.30, 0.40, 0.50]ratios = [0.8, 0.9, 1.0, 1.1, 1.2]K_sens = 90flex_matrix = np.zeros((len(vols), len(ratios)))for i, vol in enumerate(vols):    for j, ratio in enumerate(ratios):        s = K_sens * ratio / 0.9 * ratio  # Adjust S0        s_val = 100 * ratio        npv = s_val - K_sens        _, opt, _, _, _ = binomial_option_value(s_val, K_sens, vol, 0.03, 2, n=10)        flex_matrix[i, j] = opt[0, 0] - npvfig = go.Figure(data=go.Heatmap(    z=flex_matrix,    x=[f'{r:.0%}' for r in ratios],    y=[f'{v:.0%}' for v in vols],    text=np.round(flex_matrix, 1),    texttemplate='%{text}',    colorscale='YlOrRd',    colorbar_title='Flexibility<br>Value (B)'))fig.update_layout(title='Flexibility Value: Volatility × Project Value Ratio',                  xaxis_title='S₀/K Ratio (Project Value / Investment Cost)',                  yaxis_title='Volatility (σ)',                  height=400, width=600)fig.show()print("📊 민감도 분석 해석:")print("  - 변동성 ↑ → 유연성 가치 ↑ (모든 비율에서)")print("  - NPV ≈ 0 (비율 100%)일수록 유연성의 상대적 가치가 큼")print("  - NPV가 크게 양수(120%) → 기다릴 이유 감소, 유연성 가치 ↓")

### 📝 실습 과제 13-4 (25분)#### 과제: 이항 모델 파라미터 변경 분석위에서 구현한 `binomial_option_value()` 함수를 활용하여 다음을 수행하라.**단계:**1. 파라미터를 변경하여 옵션 가치 계산 (S₀=150, K=140, σ=0.25, r=0.03, T=3)2. 변동성을 10%~50%까지 변화시키며 옵션 가치 민감도 분석3. 결과를 표와 그래프로 정리---

In [ ]:
# ========== 여기서부터 코드를 작성하세요 ==========# TODO: 1. 파라미터 설정S0_my = 150  # 프로젝트 현재가치 (억원)K_my = 140   # 투자비용 (억원)sigma_my = 0.25r_my = 0.03T_my = 3# TODO: 2. 이항 모델 계산# asset_my, option_my, u_my, d_my, p_my = binomial_option_value(...)# TODO: 3. 결과 출력# print(f"전통적 NPV = {S0_my - K_my}억 원")# print(f"실물옵션 가치 = {option_my[0,0]:.2f}억 원")# TODO: 4. 변동성 민감도 분석 (σ = 0.1, 0.2, 0.3, 0.4, 0.5)# for vol in [0.1, 0.2, 0.3, 0.4, 0.5]:#     _, opt, _, _, _ = binomial_option_value(S0_my, K_my, vol, r_my, T_my, n=10)#     print(f"  σ={vol:.0%}: 옵션가치 = {opt[0,0]:.2f}억")# TODO: 5. 시각화 (Plotly 사용)# ========== 여기까지 ==========

### ✅ 실습 과제 13-4 제출란**학번:** ___________**이름:** ___________**질문 1:** S₀=150, K=140일 때 전통적 NPV와 실물옵션 가치는 각각?답: 전통적 NPV = ___억, 옵션 가치 = ___억**질문 2:** 변동성이 10%에서 50%로 증가하면 유연성 가치는 어떻게 변하는가?답: (설명)**질문 3:** 이 프로젝트에 대해 즉시 투자와 연기 중 어느 것을 권고하는가? 근거는?답: (설명)---

---## Part 5: 🔬 실습 - 단계적 투자 시뮬레이션### 실습 배경**시나리오:** 신규 기술 플랫폼 개발 (4단계 투자)앞서 구현한 `simulate_staged_investment()` 함수를 활용하여 단계적 투자를 분석하고,5단계 프로젝트로 확장한다.---

In [ ]:
# ========================================# 단계적 투자: 상세 결과 분석# ========================================# 변동성별 비교volatility_scenarios = [0.10, 0.20, 0.30, 0.40, 0.50]scenario_results = []for vol_factor in volatility_scenarios:    # 변동성을 성공률 조정으로 반영 (단순화)    np.random.seed(42)    res = simulate_staged_investment(n_sims=10000)    scenario_results.append({        'volatility': vol_factor,        'traditional_npv': res['traditional_npv'],        'option_value': res['option_value'],        'flexibility_value': res['flexibility_value']    })# 일괄 vs 단계적 비교 표print("📊 일괄 투자 vs 단계적 투자 비교")print("=" * 65)print(f"  {'투자 방식':<15} {'NPV/가치':>12} {'최대 손실':>10} {'특징'}")print("  " + "-" * 60)print(f"  {'일괄 투자':<15} {results['traditional_npv']:>10.1f}억 {'370억':>10} 초기 전액, 중단 불가")print(f"  {'단계적 투자':<15} {results['option_value']:>10.1f}억 {'20억':>10} 단계별 중단, 손실 제한")print(f"\n  유연성 가치: {results['flexibility_value']:.1f}억 원")# 시각화fig = go.Figure()labels = ['Traditional NPV\n(Lump-sum)', 'Staged Option\nValue', 'Flexibility\nValue']values = [results['traditional_npv'], results['option_value'], results['flexibility_value']]colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in values]fig.add_trace(go.Bar(x=labels, y=values, marker_color=colors,                     text=[f'{v:+.1f}B' for v in values], textposition='auto'))fig.add_hline(y=0, line_dash='dash', line_color='gray')fig.update_layout(title='Lump-sum vs Staged Investment Comparison',                  yaxis_title='Value (100M KRW)', height=400, width=600)fig.show()print("\n💡 핵심: 전통 NPV -261억(강력 기각) → 단계적 투자 옵션가치 양수(투자 가치!)")print("   1단계 20억만 투자하고, 결과에 따라 계속/중단 결정하는 것이 최적")

### 📝 실습 과제 13-5 (25분)#### 과제: 5단계 R&D 프로젝트 시뮬레이션`simulate_staged_investment()` 함수를 참고하여, **5단계** R&D 프로젝트를 시뮬레이션하라.**단계:**1. 5단계 투자 구조 정의 (각 단계의 비용, 기간, 성공률)2. 시뮬레이션 실행 (10,000회)3. 전통적 NPV와 단계적 투자 옵션 가치 비교4. 단계별 생존 퍼널 시각화5. 포트폴리오 관점 분석 (동시 추진 1/3/5/10개)---

In [ ]:
# ========== 여기서부터 코드를 작성하세요 ==========# TODO: 1. 5단계 투자 구조 정의stages_5 = [    {'name': 'Stage 1 (Idea)', 'cost': 10, 'duration': 1, 'success_rate': 0.80},    {'name': 'Stage 2 (Feasibility)', 'cost': 30, 'duration': 1, 'success_rate': 0.65},    {'name': 'Stage 3 (Prototype)', 'cost': 60, 'duration': 2, 'success_rate': 0.55},    {'name': 'Stage 4 (Pilot)', 'cost': 120, 'duration': 2, 'success_rate': 0.50},    {'name': 'Stage 5 (Launch)', 'cost': 180, 'duration': 2, 'success_rate': 0.75},]final_value_5 = 800  # 최종 성공 시 가치 (억원)# TODO: 2. 시뮬레이션 함수 수정 및 실행# (힌트: simulate_staged_investment 함수를 참고하여 5단계 버전 구현)# TODO: 3. 결과 분석# - 전통적 NPV 계산# - 단계적 투자 옵션 가치# - 단계별 생존률# TODO: 4. 시각화 (퍼널 차트 + 분포 히스토그램)# TODO: 5. 포트폴리오 분석# ========== 여기까지 ==========

### ✅ 실습 과제 13-5 제출란**학번:** ___________**이름:** ___________**질문 1:** 5단계 프로젝트의 전통적 NPV와 단계적 투자 옵션 가치는 각각?답: NPV = ___억, 옵션 가치 = ___억**질문 2:** 누적 성공률(5단계 모두 통과)은 몇 %인가?답: ____%**질문 3:** 10개 프로젝트를 동시 추진하면 최소 1개 성공 확률은?답: ____%**질문 4:** 이 분석 결과를 경영진에게 어떻게 보고하겠는가? (2-3문장)답: (설명)---

---## 🎓 강의 마무리 및 핵심 요약### 오늘 배운 내용#### 1. 실물옵션의 개념- 금융옵션의 개념을 실물 투자에 적용: **유연성에는 경제적 가치**가 있다- 확장 NPV = 전통적 NPV + 실물옵션 가치#### 2. 5가지 실물옵션 유형- **연기**(기다릴 권리), **확장**(규모 확대), **축소/포기**(손실 제한)- **전환**(변경 권리), **단계적 투자**(관문별 중단)#### 3. 이항 모델- u, d, p → 역방향 귀납법으로 옵션 가치 계산- 변동성 ↑ → 옵션 가치 ↑ (불확실성 = 기회)#### 4. 단계적 투자- 일괄 투자 NPV -261억 → 단계적 투자 옵션가치 양수!- 각 관문에서 중단 가능 → 최대 손실을 1단계로 제한#### 5. 기획에의 적용- Stage-Gate 프로세스, 전략적 유연성 설계- 포트폴리오 관점의 R&D 투자---### 핵심 메시지> **불확실성이 높을수록 유연성의 가치가 커진다.**> 전통적 NPV만으로는 유연성의 가치를 포착할 수 없다.> "기다릴 권리", "확장할 권리", "중단할 권리"에는 경제적 가치가 있으며,> 이를 정량화하는 것이 실물옵션 분석이다.---### 다음 장 예고**제14장: 실행 계획과 리스크 관리**- 13장의 단계적 투자 → **Stage-Gate 프로세스**로 실행 계획 구조화- 포기 옵션 → **리스크 대응 전략** (회피/완화/전가/수용)- 유연성 설계 → **적응적 로드맵**으로 변화 대응---### 📚 추가 학습 자료#### 참고문헌- Dixit, A.K. & Pindyck, R.S. (1994). *Investment Under Uncertainty*. Princeton University Press.- Copeland, T. & Antikarov, V. (2001). *Real Options: A Practitioner's Guide*. Texere.- Trigeorgis, L. (1996). *Real Options: Managerial Flexibility and Strategy in Resource Allocation*. MIT Press.- Hull, J.C. (2018). *Options, Futures, and Other Derivatives*. 10th ed. Pearson.#### Python 라이브러리- `scipy.stats`: 정규분포, 통계 분석- `plotly`: 인터랙티브 시각화 (옵션 페이오프, 히트맵, 퍼널)- `numpy`: 시뮬레이션, 수치 연산---**수고하셨습니다!**